# Team Challenge: TOOLBOX — Ejemplo de uso
## Dataset: Titanic

En este notebook demostramos el uso de todas las funciones del módulo `toolbox_ML.py` sobre el dataset del Titanic, que ya conocemos bien de sprints anteriores.

## 0. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# importo mi propia toolbox como si fuera una librería más
import toolbox_ML as bt

In [ ]:
# cargo el titanic — ajusta la ruta si lo tienes en otro sitio
df = pd.read_csv('./data/titanic.csv')
df.head()

## 1. `describe_df` — primer vistazo al dataframe

Sustituye al típico combo `df.info()` + `df.describe()` con una sola llamada.
Devuelve tipos, % de nulos, valores únicos y cardinalidad de cada columna.

In [ ]:
bt.describe_df(df)

**Conclusiones:**
- `Age` y `Cabin` tienen una cantidad importante de nulos.
- `PassengerId`, `Name` y `Ticket` tienen cardinalidad muy alta — no aportan como features.
- `Survived` y `Sex` son claramente binarias.

## 2. `tipifica_variables` — clasificación automática de variables

Usa umbrales de cardinalidad para sugerir si cada variable es Binaria, Categórica, Numérica Discreta o Numérica Continua.

In [ ]:
# umbral_categoria=10: si tiene menos de 10 valores únicos la trato como categórica
# umbral_continua=30: si además supera el 30% de cardinalidad la considero continua
bt.tipifica_variables(df, umbral_categoria=10, umbral_continua=30)

**Conclusiones:**
- `Fare` y `Age` quedan como Numéricas Continuas — tienen sentido para un modelo de regresión.
- `Pclass`, `SibSp`, `Parch` quedan como Numéricas Discretas o Categóricas según los umbrales.
- `Survived` y `Sex` binarias, como era de esperar.

## 3. `get_features_num_regression` — selección de features numéricas

Vamos a intentar predecir la `Fare` (precio del billete). La función devuelve las columnas numéricas con correlación significativa con el target.

In [ ]:
# sin filtro de pvalue: solo por correlación > 0.1
features_num = bt.get_features_num_regression(df, target_col='Fare', umbral_corr=0.1)
print('Features numéricas correlacionadas con Fare:', features_num)

In [ ]:
# con filtro de pvalue: además exijo significación estadística al 95%
features_num_sig = bt.get_features_num_regression(df, target_col='Fare', umbral_corr=0.1, pvalue=0.05)
print('Features numéricas significativas con Fare:', features_num_sig)

In [ ]:
# pruebo que los checks funcionan — target no numérico
bt.get_features_num_regression(df, target_col='Sex', umbral_corr=0.1)

In [ ]:
# pruebo que los checks funcionan — umbral_corr fuera de rango
bt.get_features_num_regression(df, target_col='Fare', umbral_corr=1.5)

## 4. `plot_features_num_regression` — visualización de correlaciones

Pinta pairplots de las features numéricas que pasan el filtro. Si hay muchas, las divide en grupos de máximo 5 columnas.

In [ ]:
# con columns vacío: evalúa todas las numéricas automáticamente
resultado = bt.plot_features_num_regression(df, target_col='Fare', umbral_corr=0.1, pvalue=0.05)
print('Columnas pintadas:', resultado)

In [ ]:
# pasando una lista concreta de columnas
resultado = bt.plot_features_num_regression(
    df,
    target_col='Fare',
    columns=['Age', 'SibSp', 'Parch', 'PassengerId'],
    umbral_corr=0.05
)
print('Columnas pintadas:', resultado)

## 5. `get_features_cat_regression` — selección de features categóricas

Detecta qué variables categóricas tienen relación estadísticamente significativa con el target numérico.
Usa t-test si la categórica es binaria, ANOVA si tiene más de 2 grupos.

In [ ]:
# qué categóricas tienen relación significativa con el precio del billete
features_cat = bt.get_features_cat_regression(df, target_col='Fare', pvalue=0.05)
print('Features categóricas significativas con Fare:', features_cat)

In [ ]:
# siendo más exigente con el nivel de significación
features_cat_estricto = bt.get_features_cat_regression(df, target_col='Fare', pvalue=0.01)
print('Features categóricas con pvalue < 0.01:', features_cat_estricto)

In [ ]:
# pruebo los checks — pvalue fuera de rango
bt.get_features_cat_regression(df, target_col='Fare', pvalue=1.5)

## 6. `plot_features_cat_regression` — visualización categóricas vs target

Pinta histogramas agrupados del target para cada valor de las categóricas que pasan el filtro estadístico.

In [ ]:
# histograma agrupado — todas las categóricas significativas en un solo gráfico por variable
resultado = bt.plot_features_cat_regression(df, target_col='Fare', pvalue=0.05)
print('Columnas pintadas:', resultado)

In [ ]:
# con with_individual_plot=True: un subplot por cada categoría
resultado = bt.plot_features_cat_regression(
    df,
    target_col='Fare',
    columns=['Sex', 'Embarked'],
    pvalue=0.05,
    with_individual_plot=True
)
print('Columnas pintadas:', resultado)

## Conclusiones finales

Con el dataset del Titanic y `Fare` como target de regresión:

- **Numéricas correlacionadas**: `Pclass`, `SibSp`, `Parch` muestran cierta correlación con el precio del billete.
- **Categóricas significativas**: `Sex` y `Embarked` tienen relación estadísticamente significativa con `Fare`.
- La toolbox permite hacer todo el análisis de selección de features en pocas líneas de código, con checks de entrada que evitan errores silenciosos.